# Automated coin microscope: the simulated rig

This notebook runs the coin scanner's real control code with **simulated hardware**, so you can try it with no Arduino, motors or microscope.

* `coin_scan.py` is the program that drives the real rig: it moves the stage, takes overlapping photos, removes the background, stitches the photos with Hugin and crops the finished coin.
* `rig_sim.py` swaps the hardware for pretend versions: a fake USB port running a Python copy of the Arduino firmware (`coin_stage.ino`), and a fake microscope camera that looks at a coin picture. `coin_scan.py` itself is not changed.
* `test_rig.py` checks the results against the simulator, which knows exactly where the coin really is.

**Run it:** choose **Runtime > Run all**. Setup takes about a minute, and the tests about 4 minutes.

In [ ]:
# Setup: download the code and install Hugin (the stitching program) and OpenCV.
REPO = "https://github.com/hayderbopper/coin-book"
BRANCH = "main"
!git clone -q --depth 1 -b $BRANCH $REPO coin-book
!apt-get -qq install -y hugin-tools enblend > /dev/null
!pip -q install opencv-python-headless numpy
%cd coin-book/scanner/windows

## 1. Run the automated tests

Each test runs a real `coin_scan.py` command against the simulated rig and checks the answer: does the fake Arduino reply exactly like the real firmware, does `calibrate` find the right scale and camera angle, does `scan` remove the background without cutting into the coin, and does Hugin stitch the whole coin at the right size.

In [ ]:
!python test_rig.py 2>&1 | grep -E "\.\.\. |^Ran |^OK|^FAILED|Error"

## 2. Scan a real coin, step by step

The coin is a 1999 Bulgarian 1 stotinka (16 mm), from the coin book's own scans. The settings in `rig_sim.json` are used by every command below. Try changing them and running the cells again: for example set `"camera_angle_deg": 20`, `"coin_offset_mm": [1.0, -0.5]` or `"gear_play_mm": 0.4`, and see what still works.

In [ ]:
import json
settings = {
    "coin": "../../images/bulgaria/1-stotinka/madara-rider/1999/obverse.png",
    "coin_mm": 16,
    "background": "felt",     # mat, felt or paper
    "camera_angle_deg": 3.0,  # how far the camera is turned on its mount
    "coin_offset_mm": [0, 0], # how far off-centre the coin was put down
    "gear_play_mm": 0.12,     # backlash in the motor gears
}
json.dump(settings, open("rig_sim.json", "w"), indent=2)
json.dump({"settle_s": 0.3}, open("config.json", "w"))

**Calibrate:** the stage moves 1 mm in X and Y, and the program measures how far the picture moved. The real answer is 100 px per mm with the camera turned 3°.

In [ ]:
!python rig_sim.py calibrate

**Scan:** the stage visits every spot in a grid and takes a photo. Photos on the coin's edge get the background made transparent.

In [ ]:
!rm -rf demo && python rig_sim.py scan 16 --out demo | tail -4

In [ ]:
import cv2, glob, numpy as np
import matplotlib.pyplot as plt

def show(img, title=""):
    if img.ndim == 3 and img.shape[2] == 4:  # draw transparent parts as a checkerboard
        h, w = img.shape[:2]
        yy, xx = np.mgrid[0:h, 0:w]
        board = np.where(((yy // 20 + xx // 20) % 2)[..., None] == 0, 255, 200).astype(np.float32)
        a = img[:, :, 3:] / 255
        img = (img[:, :, :3] * a + board * (1 - a)).astype(np.uint8)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis("off")

tiles = sorted(glob.glob("demo/tile_*.png"))
plt.figure(figsize=(14, 10))
for i, path in enumerate(tiles[:24]):
    plt.subplot(4, 6, i + 1)
    show(cv2.imread(path, cv2.IMREAD_UNCHANGED), path.split("/")[-1])
plt.tight_layout(); plt.show()

**Stitch and crop:** Hugin lines the photos up and blends them into one picture, then `cropfinal` finds the coin's circle and removes everything outside it.

In [ ]:
!python rig_sim.py stitch demo 2>&1 | tail -1
!python rig_sim.py cropfinal demo/coin.tif
plt.figure(figsize=(8, 8))
show(cv2.imread("demo/coin_cropped.png", cv2.IMREAD_UNCHANGED), "finished scan")
plt.show()

## 3. Talk to the simulated Arduino yourself

`import rig_sim` replaces `serial.Serial`, so this is exactly how `coin_scan.py` talks to the real board. Try other commands from the top of `coin_stage.ino`, or a move past the 24 mm limit.

In [ ]:
import importlib, sys
sys.path.insert(0, ".")
import rig_sim
import serial

port = serial.Serial("COM3", 115200, timeout=1)
print(port.readline())          # the Nano says hello when it starts
for command in ["PING", "MOVE 5 -3", "JOG 1 1", "POS", "MOVE 30 0", "HOME", "OFF"]:
    port.write((command + "\n").encode())
    print(f"{command:12} -> {port.readline().decode().strip()}")